<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c11-header.png" alt="Nextia Learning · Evaluating and Testing AI Systems" width="100%">

# Solution: Regression-test a classifier release by slice

**[Regression-test a classifier release by slice](https://learning.nextia-ai.com/courses/evals/m07/regression-test-a-classifier-release/)** · Evaluating and Testing AI Systems · Case study · [learning.nextia-ai.com](https://learning.nextia-ai.com)

A team wants to replace the comment classifier of a news site with a new version that is better on average. In this
notebook you check that claim the way a careful team does: rules and blockers written first, a frozen holdout,
overall and per-slice results with paired bootstrap intervals, and a written release decision. The data is
100,000 real public
comments from the Civil Comments archive, labelled by crowd raters for toxicity and for the groups of people they mention.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page; the notebook runs in about 3 minutes |
| **You need** | An internet connection for the setup cells. No account and no key: both model versions are trained here, on the CPU, in about a minute |
| **Before this** | [Prediction metrics](https://learning.nextia-ai.com/courses/evals/m03/prediction-metrics/), [Uncertainty and slices](https://learning.nextia-ai.com/courses/evals/m05/uncertainty-and-slices/) and [Trade-offs and regressions](https://learning.nextia-ai.com/courses/evals/m05/trade-offs-and-regressions/) |
| **Data** | Civil Comments with identity labels (CivilComments-WILDS v1.0), CC0 (public domain). A fixed sample of 100,000 comments |
| **Tested** | Python 3.12 with pandas 3.0.6, scikit-learn 1.9.1, numpy 2.5.3 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

> **Content note.** The comments are real comments from news sites. Some of them are insulting or hateful,
> including about groups of people. This notebook never prints a comment labelled toxic. It prints only a few
> harmless comments and sentences that the course wrote.

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/evals/m07/regression-test-a-classifier-release/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C11/M07-L03-regression-test-a-classifier-release/regression-test-a-classifier-release-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the chart. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the data

Run the next cell. It makes a folder `classifier-release` and downloads six files into `classifier-release/data/` from the course's labs repository: the fixed sample of comments (about 17 MB), the saved scores of both model versions, the release rules, and two small files of sentences that the course wrote. It checks the checksum (SHA-256) of each file. It needs an internet connection and no account.

You should see `Ready:` and the six files. If a copy of a file is next to this notebook, the cell uses that copy.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `classifier-release` (in Colab: the **Files** panel on the left), then run the cell again.

**Source and licence.** Civil Comments, with the toxicity and identity labels of Jigsaw and Google's Conversation AI team (Borkan et al., 2019, "Nuanced Metrics for Measuring Unintended Bias with Real Data for Text Classification", arXiv:1903.04561), in the version of the WILDS benchmark (Koh et al., 2021). Released under [CC0](https://creativecommons.org/publicdomain/zero/1.0/): "This dataset is released under CC0, as is the underlying comment text." The course took a fixed sample and changed nothing else. See `dataset.md`.

In [ ]:
# Setup: download the data files into classifier-release/data/ and check them. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C11/M07-L03-regression-test-a-classifier-release/data/"
FILES = {
    "civil_comments_sample.csv.gz": "a31eedbf043d8f866bf7cfaa1a4dfa89df9ec3d75106e916101be91b5c279576",
    "scores.csv": "625e3ae5096bfd753331f11ada6dddcb661c075c047d54f6b4ec236ede94c8a2",
    "statements.csv": "c05843098dd307ae1ad6d7a16b956bcda217d4114a2706bcbd9d5d19548a7817",
    "new_comments.csv": "851f3df61e23669fbbd7fafd2b2238b8b341dfd8787d8f7f993f730c14e87db8",
    "train.json": "94ffec2c59ae9d53590564333f126204577822c4b6337748140b71b010a2e52d",
    "release_rules.json": "43aebb6a72cdbbb3c4d15564da9cf5e502c7cfdcacac3ef4335910362a88f950"
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("classifier-release").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            try:
                urllib.request.urlretrieve(URL + name, path)
            except OSError:  # Python without certificates (python.org on macOS): try curl
                if subprocess.run(["curl", "-fsSL", "-o", str(path), URL + name]).returncode:
                    raise SystemExit(f"Could not download {URL + name}. Check your internet connection.")
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder classifier-release and run again.")
os.chdir(WORK)
print("Ready:", ", ".join(f"data/{n} ({Path('data', n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the two model versions and the measures, so the later cells stay short. It is the same code that made the numbers on the lesson page.

- `build("v1")` and `build("v2")` make the two versions (a TF-IDF pipeline with logistic regression).
- `threshold_for(dev_scores)` gives the score above which a version holds 10% of the comments.
- `compare(...)` gives recall, the benign-held rate and ROC AUC of both versions, overall and per slice, with **paired** 95% bootstrap intervals: one resample of the comments serves both versions, as in [Uncertainty and slices](https://learning.nextia-ai.com/courses/evals/m05/uncertainty-and-slices/#compare-on-the-same-cases-the-paired-difference) (2,000 resamples, seed 2026).
- `expect_hash()` checks your answers without showing them.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the two model versions, the measures and the paired bootstrap (the course's evalkit.py),
# and checks that hide the answer.
import json
import hashlib
import time

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline

GROUPS = ["male", "female", "LGBTQ", "christian", "muslim", "other_religions", "black", "white"]
SEED = 2026
CAPACITY = 0.10   # the queue holds 10% of comments
RESAMPLES = 2000


def load_sample(path):
    """The fixed sample: one row per comment; `toxic` is 1 when at least half of the raters said toxic."""
    df = pd.read_csv(path, keep_default_na=False, na_values=[""], dtype={"id": "int64"})
    return df.assign(toxic=(df["toxicity"] >= 0.5).astype(int))


def words():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=200_000)


def chars():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=5, sublinear_tf=True, max_features=300_000)


def build(version):
    """v1: words and word pairs. v2: the same plus character n-grams. Same model."""
    features = words() if version == "v1" else FeatureUnion([("words", words()), ("chars", chars())])
    return Pipeline([("features", features), ("model", LogisticRegression(C=4, solver="liblinear"))])


def run_id(version, scores):
    """A stable run ID: the version and the first 6 hex digits of the SHA-256 of the saved scores."""
    text = "\n".join(f"{s:.6f}" for s in scores)
    return f"{version}-{hashlib.sha256(text.encode()).hexdigest()[:6]}"


def score_time(pipe, texts, repeats=3):
    """Seconds to score 1,000 comments (best of a few repeats)."""
    best = float("inf")
    for _ in range(repeats):
        t = time.perf_counter()
        pipe.predict_proba(texts)
        best = min(best, time.perf_counter() - t)
    return best / len(texts) * 1000


# ------------------------------------------------------------------ measures


def threshold_for(dev_scores, capacity=CAPACITY):
    """The score above which a version holds `capacity` of the dev comments."""
    return float(np.quantile(dev_scores, 1 - capacity))


def prepare(scores, y):
    """Sort the scores once; a resample then needs only sums (fast)."""
    uniq, inv = np.unique(scores, return_inverse=True)
    return inv, len(uniq)


def weighted_auc(inv, size, y, w):
    """ROC AUC with case weights (a bootstrap resample is a set of weights). Ties count one half."""
    pos = np.bincount(inv, weights=w * y, minlength=size)
    neg = np.bincount(inv, weights=w * (1 - y), minlength=size)
    below = np.cumsum(neg) - neg
    total = pos.sum() * neg.sum()
    return float((pos * (below + 0.5 * neg)).sum() / total) if total else float("nan")


def measures(y, held, ranks, w):
    """Recall, benign-held rate and ROC AUC of one version on some comments, with weights w."""
    return {"recall": float((w * y * held).sum() / (w * y).sum()),
            "benign_held": float((w * (1 - y) * held).sum() / (w * (1 - y)).sum()),
            "auc": weighted_auc(*ranks, y, w)}


def compare(frame, base_scores, cand_scores, base_t, cand_t, seed=SEED, resamples=RESAMPLES):
    """Point values and paired 95% bootstrap intervals, overall and per slice.
    One resample of the comments serves every measure and every slice (paired)."""
    n = len(frame)
    y_all = frame["toxic"].to_numpy().astype(float)
    masks = {"all": np.ones(n, bool)} | {g: frame[g].to_numpy() == 1 for g in GROUPS}
    names = ["recall", "benign_held", "auc"]
    parts = {}
    for key, m in masks.items():
        y = y_all[m]
        parts[key] = (m, y,
                      (base_scores[m] >= base_t).astype(float), prepare(base_scores[m], y),
                      (cand_scores[m] >= cand_t).astype(float), prepare(cand_scores[m], y))

    def both(key, w):
        m, y, hb, rb, hc, rc = parts[key]
        wm = w[m]
        return measures(y, hb, rb, wm), measures(y, hc, rc, wm)

    result = {}
    for key, (m, y, *_rest) in parts.items():
        b, c = both(key, np.ones(n))
        result[key] = {"comments": int(m.sum()), "toxic": int(y.sum()), "benign": int((1 - y).sum()),
                       "baseline": b, "candidate": c,
                       "difference": {k: c[k] - b[k] for k in names}}
    draws = {key: {k: np.empty((resamples, 2)) for k in names} for key in parts}
    rng = np.random.default_rng(seed)
    for r in range(resamples):
        w = np.bincount(rng.integers(0, n, n), minlength=n).astype(float)
        for key in parts:
            b, c = both(key, w)
            for k in names:
                draws[key][k][r] = (b[k], c[k])
    for key in parts:
        result[key]["interval"] = {}
        for k in names:
            arr = draws[key][k]
            q = lambda v: [float(x) for x in np.quantile(v, [0.025, 0.975])]
            result[key]["interval"][k] = {"baseline": q(arr[:, 0]), "candidate": q(arr[:, 1]),
                                          "difference": q(arr[:, 1] - arr[:, 0])}
    return result


def fingerprint(value):
    def norm(v):
        if isinstance(v, (float, np.floating)):
            return round(float(v), 3)
        if isinstance(v, (np.integer,)):
            return int(v)
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items())}
        return v
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]


def expect_hash(what, yours, expected):
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")


def row(name, r, m):
    """One line of a comparison table: baseline, candidate, paired difference and its interval."""
    lo, hi = r["interval"][m]["difference"]
    return (f"{name:16s} {r['baseline'][m]:.3f}  {r['candidate'][m]:.3f}  {r['difference'][m]:+.3f}  "
            f"[{lo:+.3f}, {hi:+.3f}]")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data

One row is one public comment from a news site, written between 2015 and 2017. Crowd raters read each comment. Two kinds of label come from them:

- `toxicity`: the share of raters who said "toxic" or "very toxic". A comment is **toxic** here when that share is at least 0.5.
- the 8 group columns (`male` … `white`): 1 when at least half of the raters said that the comment mentions that group of people.

`part` says where each comment belongs: `train` (to train the models), `dev` (to set thresholds and look) and `holdout` (frozen, for the decision). Run the cell.

In [ ]:
comments = load_sample("data/civil_comments_sample.csv.gz")
print(comments.groupby("part").agg(comments=("id", "size"), toxic=("toxic", "sum"), toxic_rate=("toxic", "mean")).round(3))
holdout = comments[comments["part"] == "holdout"]
print("\nraters per comment (holdout): median", holdout["toxicity_annotator_count"].median(),
      "| exactly 4 raters:", round((holdout["toxicity_annotator_count"] == 4).mean(), 3),
      "| toxicity exactly 0.5:", int((holdout["toxicity"] == 0.5).sum()), "of", int(holdout["toxic"].sum()), "toxic")
print("\nholdout slices:")
print(pd.DataFrame({g: {"comments": int(holdout[g].sum()), "toxic_rate": round(holdout.loc[holdout[g] == 1, "toxic"].mean(), 3)}
                    for g in GROUPS}).T)

> **Check.** You should see 60,000 train, 10,000 dev and 30,000 holdout comments, about 11% toxic in each part. The median comment has 4 raters, and 964 toxic holdout comments sit exactly on 0.5.

**What this tells us.** About one comment in nine is toxic. Most labels come from only 4 people, so a label of 0.5 often means "2 of 4 raters". That is real noise in the labels: both versions are measured against it in the same way, so the comparison is fair, but no version can be "right" on a comment that the raters themselves split on. The slices differ a lot: comments that mention Black or white people are toxic about 31% and 29% of the time, against 8% for comments that mention no group.

## 2. The rules, written first

The team wrote its release rules before it trained any model. They are in `release_rules.json`. Run the cell and read them in order: blockers first, then the gates, then the limit.

In [ ]:
rules = json.loads(Path("data/release_rules.json").read_text())
print(rules["decision"], "\n")
for r in rules["rules_in_order"]:
    print(f"{r['id']} ({r['kind']}): {r['rule']}")
print("\nOperating point:", rules["operating_point"])

**What this tells us.** A **blocker** stops the release on its own, whatever the averages say. B1 uses 64 short, harmless sentences that the course wrote, such as "I am a gay person." B2 watches each slice: it fires only when the candidate holds clearly more harmless comments of a group, so the whole interval must be above +0.01. The gates come after the blockers: no overall loss (G1), and a gain the team can measure (G2).

## 3. The idea on a tiny example: the benign-held rate

The site holds a comment for a moderator when its score is above the threshold. **Recall** is the share of toxic comments that are held. The **benign-held rate** is the share of harmless comments that are held: the comments whose writers wait for a person for no reason. It is the false-positive rate of [Prediction metrics](https://learning.nextia-ai.com/courses/evals/m03/prediction-metrics/), read inside one slice.

A tiny slice of 10 comments: T1–T3 are toxic, B1–B7 are harmless. v1 holds T1, T2 and B1. v2 holds T1, T2, T3, B1, B2 and B3.

> **Your turn.** Compute v2's recall and benign-held rate on this slice, as fractions. Then run the check.

In [ ]:
tiny_v2 = {"recall": 3 / 3, "benign_held": 3 / 7}

In [ ]:
expect_hash("tiny_v2", tiny_v2, 'b18f856a4cf4ff54')

v1 has recall 2/3 and a benign-held rate of 1/7 = 0.14. A version can win on recall and lose on the benign-held rate. That is why the rules watch both.

## 4. Train the two versions

v1, the version on the site, reads words and word pairs. v2 adds **character n-grams**: short runs of 2 to 5 letters, so that it can match disguised words. That is the one change. Both train on the same 60,000 comments.

> **Predict.** Will v2 take longer to train than v1? About how much longer?

Run the cell. It takes about a minute.

In [ ]:
train = comments[comments["part"] == "train"]
rest = comments[comments["part"] != "train"]
models, mine = {}, rest[["id", "part"]].copy()
for version in ("v1", "v2"):
    started = time.perf_counter()
    models[version] = build(version).fit(train["comment_text"], train["toxic"])
    mine[version] = models[version].predict_proba(rest["comment_text"])[:, 1].round(6)
    print(f"{version}: trained in {time.perf_counter() - started:.0f} s")

The course saved the scores of both versions once, with stable run IDs (`v1-13f555` and `v2-8b2aeb`). The rest of the notebook uses those saved scores, so your numbers equal the page's numbers exactly. First compare your scores with the saved ones. Run the cell.

In [ ]:
scores = pd.read_csv("data/scores.csv")
train_info = json.loads(Path("data/train.json").read_text())
both = scores.merge(mine, on=["id", "part"], suffixes=("", "_mine"))
for version in ("v1", "v2"):
    t = train_info[version]["threshold"]
    flips = int(((both[version] >= t) != (both[version + "_mine"] >= t)).sum())
    print(f"{train_info[version]['run_id']}: largest difference {abs(both[version] - both[version + '_mine']).max():.6f}, "
          f"held/shown decisions that differ: {flips} of {len(both)}")

> **Check.** On the course's computer, the largest difference is 0.000000 and no decision differs. Another computer can give tiny differences in the last digits; a handful of comments right at a threshold may then flip. That is why a team compares runs by their saved outputs and run IDs.

## 5. Thresholds and a first look on dev

The site's moderators can read 10% of the comments. So each version gets its own threshold: the score above which it holds 10% of the **dev** comments. Thresholds come from dev, never from the holdout. A trivial baseline helps to read the numbers: holding 10% of comments **at random** would catch about 10% of the toxic ones.

Run the cell. It prints both thresholds and the dev comparison: baseline, candidate, the paired difference and its 95% interval.

In [ ]:
t1, t2 = train_info["v1"]["threshold"], train_info["v2"]["threshold"]
print("thresholds from dev: v1", t1, "| v2", t2)
dev = comments[comments["part"] == "dev"].merge(scores[["id", "v1", "v2"]], on="id")
dev_result = compare(dev, dev["v1"].to_numpy(), dev["v2"].to_numpy(), t1, t2)
print("\nrecall (dev)        v1     v2     diff    95% interval")
print(row("all", dev_result["all"], "recall"))
print("\nROC AUC (dev)")
print(row("all", dev_result["all"], "auc"))

**What this tells us.** On dev, v2 catches a few more toxic comments: recall 0.534 → 0.549, but the interval [-0.007, +0.036] includes 0. Its ROC AUC is higher, with an interval above 0. Both versions do far better than the random baseline (0.10). The team does not change any rule after this look. The decision waits for the holdout.

## 6. The holdout, once

> **Predict.** On the holdout, will v2's overall recall be better than v1's, with an interval above 0? Will every slice improve?

Run the cell. It compares the two versions on the 30,000 holdout comments, overall and in each of the 8 slices.

In [ ]:
hold = holdout.merge(scores[["id", "v1", "v2"]], on="id")
result = compare(hold, hold["v1"].to_numpy(), hold["v2"].to_numpy(), t1, t2)
for m, title in (("recall", "recall: toxic comments held"), ("benign_held", "benign held: harmless comments held")):
    print(f"\n{title}\n{'':16s} v1     v2     diff    95% interval")
    for key in ["all", *GROUPS]:
        print(row(key, result[key], m))

**What this tells us.** Overall, v2 is better: recall 0.512 → 0.532, +0.019 [+0.008, +0.031], and it holds fewer harmless comments (0.048 → 0.044). Most slices improve or stay level. One does not: in the **LGBTQ** slice, v2 holds more harmless comments, 0.160 → 0.196, +0.036 [+0.015, +0.057]. The whole interval is above +0.01.

The chart shows the benign-held differences with their intervals. Run the cell.

In [ ]:
import matplotlib.pyplot as plt
keys = ["all", *GROUPS]
diff = [result[k]["difference"]["benign_held"] for k in keys]
low = [result[k]["interval"]["benign_held"]["difference"][0] for k in keys]
high = [result[k]["interval"]["benign_held"]["difference"][1] for k in keys]
fig, ax = plt.subplots(figsize=(7, 4))
y = range(len(keys))[::-1]
ax.errorbar(diff, y, xerr=[[d - l for d, l in zip(diff, low)], [h - d for d, h in zip(diff, high)]], fmt="o", capsize=3)
ax.axvline(0, color="grey", linestyle="--")
ax.axvline(0.01, color="tab:orange", linestyle=":", label="B2 line: +0.01")
ax.set_yticks(list(y), keys)
ax.set_xlabel("benign-held rate, v2 − v1 (paired, 95% interval)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. The rules, in order

Now apply the rules exactly as written. B1 first: which of the course's harmless sentences does each version hold? Run the cell.

In [ ]:
statements = pd.read_csv("data/statements.csv")
statements["held_v1"], statements["held_v2"] = statements["v1"] >= t1, statements["v2"] >= t2
print("held by v1:", int(statements["held_v1"].sum()), "| held by v2:", int(statements["held_v2"].sum()), "of", len(statements))
print("\nheld by v2 only (B1):")
print(statements[statements["held_v2"] & ~statements["held_v1"]][["text", "v1", "v2"]].to_string(index=False))
print("\nheld by v1 only:")
print(statements[statements["held_v1"] & ~statements["held_v2"]][["text", "v1", "v2"]].to_string(index=False))

**What this tells us.** v1 holds 11 of the 64 harmless sentences, v2 holds 6. So v2 is better on this set as a whole. But it holds one sentence that v1 shows: "I am a Muslim person." The rule says that one sentence is enough, and the team wrote it before it saw any result. B1 fires.

> **Your turn.** Write B2 as code. Make a list `b2_slices` of the slices (from `GROUPS`) whose paired interval of the benign-held difference has a lower end above 0.01. Use `result[g]["interval"]["benign_held"]["difference"]`, which is `[low, high]`.

In [ ]:
b2_slices = [g for g in GROUPS if result[g]["interval"]["benign_held"]["difference"][0] > 0.01]
print(b2_slices)

In [ ]:
expect_hash("b2_slices", b2_slices, '9ef436d999db9f58')

Then the gates and the limit. Run the cell. It applies every rule and prints the decision.

In [ ]:
low_recall = result["all"]["interval"]["recall"]["difference"][0]
b1 = statements.loc[statements["held_v2"] & ~statements["held_v1"], "text"].tolist()
b2 = [g for g in GROUPS if result[g]["interval"]["benign_held"]["difference"][0] > 0.01]
checks = {
    "B1 identity statements": "BLOCKER " + str(b1) if b1 else "pass",
    "B2 slice regression": "BLOCKER " + str(b2) if b2 else "pass",
    "G1 no overall loss": "pass" if low_recall >= -0.01 else "fail",
    "G2 a measurable gain": "pass" if low_recall > 0 else "fail",
    "O1 scoring time": "pass" if train_info["v2"]["ms_per_1000"] < 1000 else "fail",
}
for name, outcome in checks.items():
    print(f"{name:24s} {outcome}")
print("\nDecision:", "SHIP" if all(v == "pass" for v in checks.values()) else "DO NOT SHIP")

> **Check.** You should see two blockers and `Decision: DO NOT SHIP`.

## 8. Read the cases behind the slice

An interval says how sure the team is about a difference. It does not say which comments changed. In the LGBTQ slice, 730 holdout comments are harmless. Run the cell to count which ones changed between the versions, and which words they contain.

In [ ]:
lg = hold[(hold["LGBTQ"] == 1) & (hold["toxic"] == 0)].assign(h1=lambda d: d["v1"] >= t1, h2=lambda d: d["v2"] >= t2)
newly, released = lg[lg["h2"] & ~lg["h1"]], lg[~lg["h2"] & lg["h1"]]
print("harmless LGBTQ comments:", len(lg), "| held by both:", int((lg["h1"] & lg["h2"]).sum()),
      "| newly held by v2:", len(newly), "| released by v2:", len(released))
print("newly held with 30% or more raters saying toxic:", int((newly["toxicity"] >= 0.3).sum()))
for word in ["gay", "lesbian", "homosexual", "transgender"]:
    has = lambda s: int(s.str.contains(rf"\b{word}", case=False).sum())
    print(f"{word:12s} newly held {has(newly['comment_text']):3d} | released {has(released['comment_text']):3d}")
example = hold[hold["id"] == 267034]
print("\nOne harmless comment that v2 now holds:\n", example["comment_text"].iloc[0])
print("raters:", int(example["toxicity_annotator_count"].iloc[0]), "| toxicity", round(example["toxicity"].iloc[0], 3),
      "| v1", round(example["v1"].iloc[0], 3), "| v2", round(example["v2"].iloc[0], 3))

**What this tells us.** v2 newly holds 45 harmless LGBTQ comments and releases 19. The new holds lean on words: "homosexual" appears in 22 of them, "transgender" in 8, "lesbian" in 5, and none of the released comments has "transgender" or "lesbian". The character n-grams of these words carry some of the toxicity that v2 learned. 14 of the 45 are borderline (30% or more of raters said toxic), so not every one is a clear mistake; the rest are.

## 9. The baseline is not fair either

The blockers stop a release that makes things **worse**. They do not say that v1 is good. Compare the harmless comments that mention no group with those that mention at least one.

> **Your turn.** Compute `no_group_rate`: v1's benign-held rate on the holdout comments that mention **none** of the 8 groups (harmless comments only). Round to 3 decimals.

In [ ]:
none = hold[(hold[GROUPS].sum(axis=1) == 0) & (hold["toxic"] == 0)]
no_group_rate = round(float((none["v1"] >= t1).mean()), 3)
print(no_group_rate)

In [ ]:
expect_hash("no_group_rate", no_group_rate, '71291a2091789c96')

v1 holds 2.1% of harmless comments that mention no group, and 9.3% of harmless comments that mention at least one. For Black and white, it is 29.6% and 27.9%. People who write about their own identity wait for a moderator far more often. A regression test protects the present; it does not fix it.

## 10. A prediction on new comments

The trained models can score any new comment. These six were written by the course. Run the cell.

In [ ]:
new = pd.read_csv("data/new_comments.csv")[["text"]]
for version, t in (("v1", t1), ("v2", t2)):
    new[version] = models[version].predict_proba(new["text"])[:, 1].round(3)
    new["held_" + version] = new[version] >= t
print(new.to_string(index=False))

**What this tells us.** Both versions show the four harmless comments and hold the insult. Neither holds "What an 1d1ot. Go away.", although disguised words were the reason for v2. One example proves nothing, but it is worth a test case in the next evaluation.

## 11. Change one thing

The decision is made; this is for learning only. Suppose the moderators could read only **5%** of the comments. The thresholds move up, and both versions hold fewer comments.

> **Predict.** With a 5% queue, does B2 still fire for the LGBTQ slice?

> **Your turn.** Compute the thresholds at 5% from the dev scores (`threshold_for(dev["v1"], 0.05)`), run `compare` on the holdout again, and make `b2_at_5` the list of slices that B2 would flag.

In [ ]:
t1_5, t2_5 = threshold_for(dev["v1"], 0.05), threshold_for(dev["v2"], 0.05)
result_5 = compare(hold, hold["v1"].to_numpy(), hold["v2"].to_numpy(), t1_5, t2_5)
b2_at_5 = [g for g in GROUPS if result_5[g]["interval"]["benign_held"]["difference"][0] > 0.01]
print(b2_at_5)
print(row("LGBTQ", result_5["LGBTQ"], "benign_held"))
print(row("all", result_5["all"], "recall"))

In [ ]:
expect_hash("b2_at_5", b2_at_5, '4f53cda18c2baa0c')

With a 5% queue, the LGBTQ difference is +0.004 [-0.009, +0.018], and B2 flags no slice. Overall recall changes by +0.011 [+0.000, +0.022]. The capacity is part of the operating point: a result holds for the operating point it was measured at.

## 12. The failure case: one number

A team that reads only the overall ROC AUC sees 0.887 → 0.902, +0.014 [+0.012, +0.017], and ships. The interval is narrow and above 0, so the number is not noise. It is still the wrong question: it averages over 30,000 comments, and the LGBTQ slice is 999 of them. Ask the slice question and the blocker question before the average.

## Recap

- The rules came first: blockers, gates, a limit, and the operating point, written before any model existed.
- Both versions were compared on the same frozen holdout, overall and per slice, with paired bootstrap intervals.
- v2 is better on average, and it fails two blockers: a new hold of "I am a Muslim person." and more harmless LGBTQ comments held. **Do not ship.**
- The baseline also holds harmless comments about groups far more often than others. The decision protects users from a regression; it does not make v1 fair.

Write the decision in your own words on the lesson page, then go on to the final assignment.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/evals/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/evals/m07/regression-test-a-classifier-release/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The comments are real public comments (CC0); the course never prints one labelled toxic.